# Vicuña · T11: ventanas con huecos de horas

Misma red y configuración del ensamble enviado en T08 (pérdida `rmse`, 25 épocas, pesos finales, HR recortada a 0–100, 5 semillas). Solo cambian las ventanas de **entrenamiento**:

- `todas`: las horas anteriores al 16 jun, con las incompletas rellenadas (como hasta ahora).
- `completas`: solo las que tienen t-2, t-1 y t.

La climatología y la normalización salen de las mismas horas en las dos ramas. La validación son las 336 h del 16–30 jun (comparable con T02 y T08); aparte se reportan las completas. En test no se descarta ninguna hora.

**Regla fijada antes de correr:** se adopta `completas` salvo que empeore `score_oficial` en las 5 semillas, o que el ensamble caiga más que la desviación entre semillas de `todas`. En esos casos se discute en el PR.

No escribe pesos ni submission.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
# COPIA EXACTA de train/ventanas.py: no editar aquí
"""
Ventanas (t-2, t-1, t) del GFS sin cruzar huecos de horas (T11 · HU-09). Solo NumPy.

La data del concurso tiene huecos de horas y de días. Una ventana está completa si existen las k horas
consecutivas que terminan en t. En las incompletas, la hora que falta se rellena con la hora t (igual que
el starter): sirven para predecir horas que Kaggle exige, pero se excluyen del entrenamiento.

Las horas previas se buscan por fecha, no por posición: nunca entra una hora del otro lado de un hueco.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_ventanas.py comprueba que ambas sean iguales.
"""
from datetime import datetime, timedelta

import numpy as np


def indices_ventanas(times, k=3):
    """
    Índices de las horas (t-(k-1), …, t) de cada marca 'YYYYMMDDHH'.

    Devuelve (idx, completa): idx es (N, k), con el índice de t donde falta una hora previa;
    completa es una máscara (N,), True si las k horas existen.
    """
    fechas = [datetime.strptime(str(t), "%Y%m%d%H") for t in times]
    pos = {f: i for i, f in enumerate(fechas)}
    if len(pos) != len(fechas):
        raise ValueError("Hay horas repetidas en times.")
    idx = np.empty((len(fechas), k), np.int64)
    completa = np.ones(len(fechas), bool)
    for i, f in enumerate(fechas):
        for j, atras in enumerate(range(k - 1, -1, -1)):
            previa = pos.get(f - timedelta(hours=atras))
            if previa is None:
                completa[i] = False
                previa = i
            idx[i, j] = previa
    return idx, completa


def armar_ventanas(X, times, k=3):
    """Ventanas (N, k, ...) en float32 y la máscara de las completas."""
    if len(X) != len(times):
        raise ValueError(f"X tiene {len(X)} horas y times {len(times)}.")
    idx, completa = indices_ventanas(times, k)
    return np.asarray(X, np.float32)[idx], completa

In [ ]:
Wtr, comp_tr = armar_ventanas(Xtr, ttr)
Wte, comp_te = armar_ventanas(Xte, tte)
trn, val = corte_temporal(ttr)

conteo = {}
for nombre, mascara, completas in (('entrenamiento', trn, comp_tr), ('validacion', val, comp_tr),
                                   ('train_total', trn | val, comp_tr), ('test', np.ones(len(tte), bool), comp_te)):
    conteo[nombre] = dict(antes=int(mascara.sum()), completas=int((mascara & completas).sum()),
                          incompletas=int((mascara & ~completas).sum()))
print(pd.DataFrame(conteo).T.to_string())

In [ ]:
import json, platform
assert torch.cuda.is_available(), 'Este notebook necesita GPU'
dev = 'cuda'
QUICK = False                      # True: prueba corta de que todo corre
EPOCAS = 2 if QUICK else 25
SEMILLAS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
PERDIDA = 'rmse'
print(torch.cuda.get_device_name(0), '·', platform.python_version(), np.__version__, torch.__version__)

In [ ]:
# Coordenadas de la grilla WRF dentro de la grilla GFS, para grid_sample
glat, glon = st['gfs_lats'], st['gfs_lons']            # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']            # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1,70,35,2)
DEM = torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32, device=dev)[None, None]

class Net(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(
            nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, GRID.expand(B, -1, -1, -1), align_corners=True)   # (B,w,70,35)
        f = torch.cat([f, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

In [ ]:
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

def datos_de(base, usadas):
    """Normalización y climatología con las horas de `base`; ventanas de entrenamiento de `usadas`."""
    mu_x = Xtr[base].mean((0, 1, 2)); sd_x = Xtr[base].std((0, 1, 2))
    clim = climatologia(Ytr[base], ttr[base])
    anom_sd = (Ytr[base] - clim[horas(ttr[base])]).std((0, 1, 2))
    idx = np.where(usadas)[0]
    h = to_t(horas(ttr[idx])).long()
    CLIM, ASD = to_t(clim), to_t(anom_sd)
    x = to_t((Wtr[idx] - mu_x) / sd_x)
    y = (to_t(Ytr[idx]) - CLIM[h]) / ASD
    return {'x': x, 'y': y, 'h': h, 'CLIM': CLIM, 'ASD': ASD,
            'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd)}

def perdida(nombre, pred, y, ASD):
    if nombre == 'mse':                       # starter: las 4 anomalías normalizadas pesan igual
        return F.mse_loss(pred, y)
    mse = ((pred - y) ** 2).mean(dim=(0, 1, 2)) * (ASD / ESC) ** 2   # por variable, en unidades de la métrica
    if nombre == 'wmse':
        return mse.mean()
    if nombre == 'rmse':                      # 1 - score del lote
        return torch.sqrt(mse + 1e-8).mean()
    raise ValueError(nombre)

def entrenar(D, epocas, nombre_perdida, semilla, lote=32):
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = Net().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    n = len(D['x']); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            b = orden[s:s + lote]
            loss = perdida(nombre_perdida, model(D['x'][b], D['h'][b].float()), D['y'][b], D['ASD'])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def predecir_lote(model, D, W, t, bs=128):
    s = D['stats']; model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            x = to_t((W[i:i + bs] - s['mu_x']) / s['sd_x'])
            h = to_t(horas(t[i:i + bs])).long()
            out.append((D['CLIM'][h] + model(x, h.float()) * D['ASD']).cpu().numpy())
    return np.concatenate(out)

def recortar_hr(P):
    P = P.copy(); P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P):
    r = validar(P, Ytr, ttr)
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
ramas = {'todas': trn, 'completas': trn & comp_tr}
assert comp_tr[ramas['completas']].all()
horas_3h = np.isin(horas(ttr[val]), HORAS_EVALUADAS)
sel = comp_tr[val] & horas_3h          # horas evaluadas con ventana completa

def medir_completas(P):
    r = rmse_por_variable(P[sel], Ytr[val][sel])
    return {'score_oficial': score(r, ESCALAS_OFICIALES), 'rmse_3h': {v: float(x) for v, x in zip(VARS, r)}}

resultados = {}
for rama, usadas in ramas.items():
    D = datos_de(trn, usadas)
    print(f'--- {rama}: {len(D["x"])} ventanas de entrenamiento', flush=True)
    suma, solas, acumulado = 0, [], []
    for n, semilla in enumerate(SEMILLAS, 1):
        t0 = time.time()
        P = recortar_hr(predecir_lote(entrenar(D, EPOCAS, PERDIDA, semilla), D, Wtr[val], ttr[val]))
        suma = suma + P
        solas.append(dict(semilla=semilla, **medir(P), completas=medir_completas(P)))
        acumulado.append(medir(suma / n)['score_oficial'])
        linea(f'{rama} s{semilla} ({time.time() - t0:.0f}s)', solas[-1])
    E = suma / len(SEMILLAS)
    resultados[rama] = dict(ventanas=len(D['x']), solas=solas, acumulado=acumulado,
                            ensamble=dict(**medir(E), completas=medir_completas(E)))
    linea(f'{rama} ensamble', resultados[rama]['ensamble'])
    del D; torch.cuda.empty_cache()

In [ ]:
def serie(rama):
    return np.array([s['score_oficial'] for s in resultados[rama]['solas']])

dif = serie('completas') - serie('todas')
desv = float(serie('todas').std(ddof=1))
dif_ens = resultados['completas']['ensamble']['score_oficial'] - resultados['todas']['ensamble']['score_oficial']
peor_en_todas = bool((dif < 0).all())
cae_ensamble = bool(-dif_ens > desv)

filas = [dict(rama=r, que=que, score=m['score_oficial'], score_completas=m['completas']['score_oficial'], **m['rmse_3h'])
         for r, res in resultados.items()
         for que, m in [(f"semilla {s['semilla']}", s) for s in res['solas']] + [('ensamble', res['ensamble'])]]
print(pd.DataFrame(filas).round(5).to_string(index=False))
print(f'Diferencia por semilla (completas - todas): {np.round(dif, 5)} · media {dif.mean():.5f}')
print(f'Diferencia del ensamble: {dif_ens:.5f} · desviación entre semillas de "todas": {desv:.5f}')
print('Decisión:', 'DISCUTIR en el PR' if peor_en_todas or cae_ensamble else 'ADOPTAR la exclusión')
if QUICK:
    print('QUICK=True: estos números no valen para decidir.')
json.dump({'quick': QUICK, 'conteo': conteo, 'config': dict(perdida=PERDIDA, epocas=EPOCAS, semillas=SEMILLAS),
           'resultados': resultados,
           'decision': dict(dif_semillas=dif.tolist(), dif_ensamble=dif_ens, desv_todas=desv,
                            peor_en_todas=peor_en_todas, cae_ensamble=cae_ensamble),
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__,
                         'gpu': torch.cuda.get_device_name(0)}}, open('resultados_t11.json', 'w'), indent=1)